## RCoxNet vs baseline models — C-index comparison (Figure 4)

Reproduces Table 1 and Figure 4 from the paper.

Reads pre-computed per-repeat C-index results from `results/comparison_all_models.csv`,
which was generated by running `python scripts/run_benchmarks.py`.

Set `show_dots=True` in the last two cells to toggle jitter dots on the boxplot.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.font_manager as fm
from scipy.stats import wilcoxon

ROOT = os.getcwd()

In [ ]:
avail = {f.name for f in fm.fontManager.ttflist}
mpl.rcParams['font.sans-serif'] = ['Arial' if 'Arial' in avail else 'Liberation Sans']
mpl.rcParams['font.family']    = 'sans-serif'
mpl.rcParams['pdf.fonttype']   = 42
mpl.rcParams['ps.fonttype']    = 42
mpl.rcParams['font.size']         = 7
mpl.rcParams['axes.titlesize']    = 8
mpl.rcParams['axes.labelsize']    = 7
mpl.rcParams['xtick.labelsize']   = 7
mpl.rcParams['ytick.labelsize']   = 7
mpl.rcParams['legend.fontsize']   = 6.5
mpl.rcParams['axes.linewidth']    = 0.8
mpl.rcParams['xtick.major.width'] = 0.8
mpl.rcParams['ytick.major.width'] = 0.8
mpl.rcParams['xtick.major.size']  = 3
mpl.rcParams['ytick.major.size']  = 3

### Load results

In [ ]:
df = pd.read_csv(os.path.join(ROOT, 'results', 'comparison_all_models.csv'))
print(df.shape)
df.head()

In [ ]:
# 20 repeats per cancer per model
df.groupby(['cancer', 'model']).size().unstack().fillna(0).astype(int)

### Table 1 — mean ± std C-index

In [ ]:
CANCERS     = ['BRCA', 'GBM', 'LUNG', 'OV']
MODEL_ORDER = ['RCoxNet', 'Cox-EN', 'Cox-nnet', 'SurvivalNet', 'DeepSurv', 'DeepHit']

smry = df.groupby(['cancer', 'model'])['test_c_index'].agg(['mean', 'std']).reset_index()
smry['val'] = smry.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)

tbl = smry.pivot(index='model', columns='cancer', values='val').reindex(index=MODEL_ORDER, columns=CANCERS)
print(tbl.to_string())

In [ ]:
# check results match paper Table 1
paper = {
    ('BRCA','RCoxNet'):(0.807,0.045), ('BRCA','Cox-EN'):(0.723,0.074),
    ('BRCA','Cox-nnet'):(0.771,0.039), ('BRCA','SurvivalNet'):(0.764,0.045),
    ('BRCA','DeepSurv'):(0.782,0.038), ('BRCA','DeepHit'):(0.625,0.086),
    ('GBM','RCoxNet'):(0.704,0.042),  ('GBM','Cox-EN'):(0.646,0.050),
    ('GBM','Cox-nnet'):(0.658,0.053), ('GBM','SurvivalNet'):(0.646,0.051),
    ('GBM','DeepSurv'):(0.667,0.050), ('GBM','DeepHit'):(0.560,0.081),
    ('LUNG','RCoxNet'):(0.750,0.040), ('LUNG','Cox-EN'):(0.646,0.040),
    ('LUNG','Cox-nnet'):(0.699,0.047),('LUNG','SurvivalNet'):(0.682,0.043),
    ('LUNG','DeepSurv'):(0.722,0.035),('LUNG','DeepHit'):(0.616,0.062),
    ('OV','RCoxNet'):(0.668,0.037),   ('OV','Cox-EN'):(0.630,0.043),
    ('OV','Cox-nnet'):(0.657,0.049),  ('OV','SurvivalNet'):(0.631,0.046),
    ('OV','DeepSurv'):(0.636,0.067),  ('OV','DeepHit'):(0.532,0.048),
}

ok = True
for (c, m), (em, es) in paper.items():
    row = smry[(smry.cancer == c) & (smry.model == m)]
    gm, gs = row['mean'].values[0], row['std'].values[0]
    if abs(gm - em) > 0.0015 or abs(gs - es) > 0.0015:
        print(f'MISMATCH {c}/{m}: got {gm:.3f}±{gs:.3f}  expected {em:.3f}±{es:.3f}')
        ok = False

if ok:
    print(f'All {len(paper)} values match paper Table 1.')

### Figure 4 — boxplot comparison with significance brackets

In [ ]:
CANCER_LABELS = {
    'BRCA': 'Breast (BRCA)',
    'GBM':  'Glioblastoma (GBM)',
    'LUNG': 'Lung (LUNG)',
    'OV':   'Ovarian (OV)',
}

COLORS = {
    'Cox-EN':      '#7A9CB8',
    'Cox-nnet':    '#A88BBF',
    'SurvivalNet': '#C9A84C',
    'DeepSurv':    '#74A89E',
    'DeepHit':     '#B87060',
    'RCoxNet':     '#6A9E6E',
}

BASELINES = ['Cox-EN', 'Cox-nnet', 'SurvivalNet', 'DeepSurv', 'DeepHit']

In [ ]:
def _sig(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def _draw_brackets(ax, pairs, top_y, dy=0.028, tip=0.010):
    # pairs sorted by span length so shorter brackets sit closer to the data
    for x1, x2, lbl in pairs:
        sig  = lbl != 'ns'
        col  = '#1a1a1a' if sig else '#999999'
        h    = top_y + tip
        ax.plot([x1, x1, x2, x2],
                [h, h + dy*0.45, h + dy*0.45, h],
                lw=0.9 if sig else 0.7, color=col,
                linestyle='-' if sig else '--',
                clip_on=False, zorder=6, solid_capstyle='round')
        ax.text((x1+x2)/2, h + dy*0.45 + 0.005, lbl,
                ha='center', va='bottom',
                fontsize=6.5 if sig else 5.5,
                fontweight='bold' if sig else 'normal',
                fontstyle='normal' if sig else 'italic',
                color=col, clip_on=False)
        top_y += dy


def plot_figure4(data, show_dots=True):
    models = [m for m in ['Cox-EN','Cox-nnet','SurvivalNet','DeepSurv','DeepHit','RCoxNet']
              if m in data['model'].unique()]

    # sort by median C-index so the best model ends up on the right
    models = sorted(models, key=lambda m: np.median(data[data.model==m]['test_c_index'].values))

    n   = len(models)
    ylo = max(0.35, data['test_c_index'].min() - 0.06)
    yhi = min(1.02, data['test_c_index'].max() + 0.06 + 5*0.038)

    fig, axes = plt.subplots(1, 4, figsize=(7.09, 4.4), dpi=150, sharey=True)
    fig.subplots_adjust(wspace=0.06, left=0.10, right=0.985, top=0.76, bottom=0.22)

    for ci, (ax, cancer) in enumerate(zip(axes, CANCERS)):
        vals   = [data[(data.cancer==cancer) & (data.model==m)]['test_c_index'].values
                  for m in models]
        colors = [COLORS[m] for m in models]

        ax.axhline(0.5, color='#cccccc', lw=0.6, linestyle=':', zorder=1)

        bp = ax.boxplot(vals, patch_artist=True, widths=0.52,
                        medianprops=dict(color='#111111', linewidth=1.4, zorder=5),
                        whiskerprops=dict(color='#606060', linewidth=0.75),
                        capprops=dict(color='#606060', linewidth=0.75),
                        flierprops=dict(marker='o', markersize=1.8,
                                        markerfacecolor='#aaaaaa',
                                        markeredgewidth=0, alpha=0.6),
                        showfliers=True, zorder=2)

        for box, col in zip(bp['boxes'], colors):
            box.set_facecolor(col)
            box.set_alpha(0.85)
            box.set_linewidth(0.5)
            box.set_edgecolor('#555555')

        ri = models.index('RCoxNet')
        bp['boxes'][ri].set_edgecolor('#2e5e32')
        bp['boxes'][ri].set_linewidth(0.9)

        if show_dots:
            np.random.seed(42)
            for i, (d, col) in enumerate(zip(vals, colors)):
                jx = np.random.uniform(-0.15, 0.15, len(d))
                ax.scatter(i+1+jx, d, color=col, s=4.5, alpha=0.40,
                           edgecolors='none', zorder=3)

        # Wilcoxon one-sided test: RCoxNet vs each baseline
        rcox_v   = vals[ri]
        top_data = max(v.max() for v in vals if len(v))
        pairs = []
        for bl in BASELINES:
            if bl not in models: continue
            bi = models.index(bl)
            try:
                _, p = wilcoxon(rcox_v, vals[bi], alternative='greater')
            except Exception:
                p = 1.0
            pairs.append((bi+1, ri+1, _sig(p), abs(ri-bi)))
        pairs.sort(key=lambda x: x[3])
        _draw_brackets(ax, [(a,b,l) for a,b,l,_ in pairs], top_y=top_data)

        ax.set_title(CANCER_LABELS[cancer], fontsize=7.5, fontweight='bold', pad=5)
        ax.set_xticks(range(1, n+1))
        ax.set_xticklabels(models, rotation=40, ha='right',
                           fontsize=6.5, rotation_mode='anchor')
        ax.set_ylim(ylo, yhi)
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.10))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.05))
        ax.tick_params(axis='y', which='minor', length=2, width=0.5)
        ax.tick_params(axis='both', which='major', width=0.8, length=3, pad=2)

        if ci == 0:
            ax.set_ylabel('C-index', fontsize=7, labelpad=4)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if ci > 0:
            ax.spines['left'].set_visible(False)
            ax.tick_params(axis='y', which='both', left=False)
        ax.grid(False)

    handles = [mpatches.Patch(facecolor=COLORS[m], edgecolor='none', label=m)
               for m in models]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.54, -0.02),
               ncol=n, frameon=False, handlelength=0.9, handleheight=0.7,
               columnspacing=0.8, handletextpad=0.4, borderpad=0, fontsize=6.5)
    return fig

In [ ]:
fig = plot_figure4(df, show_dots=True)
fig.savefig(os.path.join(ROOT, 'results', 'Figure4_comparison.pdf'),
            bbox_inches='tight', format='pdf')
plt.show()

In [ ]:
fig2 = plot_figure4(df, show_dots=False)
fig2.savefig(os.path.join(ROOT, 'results', 'Figure4_comparison_no_dots.pdf'),
            bbox_inches='tight', format='pdf')
plt.show()

### Per-repeat raw numbers

In [ ]:
pd.set_option('display.max_rows', 100)
pivot = (
    df
    .assign(model=pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True))
    .sort_values(['cancer', 'model', 'repeat'])
    .pivot_table(index=['cancer','repeat'], columns='model', values='test_c_index')
    .round(4)
)
pivot